**Sample ID**: 770




**Query**: Check Gmail for lab results from the last 7 days and send them as Slack messages in channel "test-review" for a quick review.




**DB Type**: Base Case




**Case Description**: At least one email exists with the lab results in the last 7 days explicitly containing the phrase: 'lab results' in the subject. The channel "test-review" exists on Slack and the email body needs to be sent to the channel as message.




**Global/Context Variables**:


- target_slack_channel_name = "test-review"
- current_date = "2023-04-16"




**APIs**:

- gmail
- slack


# Set Up

## Download relevant files

In [1]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [2]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [5]:
import gmail
import slack
import base64
from datetime import timedelta, datetime
import random

# Load the default Gmail DB
gmail.SimulationEngine.db.load_state("/content/DBs/GmailDefaultDB.json")
slack.SimulationEngine.db.load_state("/content/DBs/SlackDefaultDB.json")

target_slack_channel_name = "test-review"
current_date = "2023-04-16"

# Pick a random date within the past week
days_ago = random.randint(0, 6)  # Any day from 0 to 6 days ago
date_in_past_week = datetime.strptime(current_date, "%Y-%m-%d") - timedelta(days=days_ago)
# Format date for Gmail (ISO 8601 format)
date_in_past_week_iso = date_in_past_week.strftime("%Y-%m-%dT%H:%M:%SZ")

#--------------GMAIL DB--------------#
# Define lab result body texts
lab_body_0 = """
Your temporary access code is ABCDEF.
Reset your credentials to access your account.
""".strip()
lab_body_1 = """
Patient Name: John Smith
Date: 2025-04-09
Abnormal Values: High glucose levels
""".strip()

lab_body_2 = """
Patient Name: Emily Davis
Date: 2025-04-08
Abnormal Values: Low hemoglobin
""".strip()

lab_body_3 = """
Binance: Your BTC buy order was filled.
Price: $29,500
Quantity: 0.05 BTC
""".strip()

# Insert the messages
a = gmail.insert_message(userId="me", msg={
    "sender": "no-reply@sheinlab.com",
    "recipient": "user@example.com",
    "subject": "Reset Credentials for Ashton Hall",
    "body": lab_body_0,
    "raw": base64.urlsafe_b64encode(f"Subject: Reset Credentials for Ashton Hall\n\n{lab_body_0}".encode("utf-8")).decode("utf-8"),
    'date': date_in_past_week_iso,
    "labelIds": ["INBOX", "UNREAD"]
})

x = gmail.insert_message(userId="me", msg={
    "sender": "me",
    "recipient": "user@example.com",
    "subject": "Lab Results for Patient - John Smith",
    "body": lab_body_1,
    "raw": base64.urlsafe_b64encode(f"Subject: Lab Results for Patient - John Smith\n\n{lab_body_1}".encode("utf-8")).decode("utf-8"),
    'date': date_in_past_week_iso,
    "labelIds": ["INBOX", "UNREAD"]
})

y = gmail.insert_message(userId="me", msg={
    "sender": "me",
    "recipient": "user@example.com",
    "subject": "Lab Results for Patient - Emily Davis",
    "body": lab_body_2,
    "raw": base64.urlsafe_b64encode(f"Subject: Lab Results for Patient - Emily Davis\n\n{lab_body_2}".encode("utf-8")).decode("utf-8"),
    'date': date_in_past_week_iso,
    "labelIds": ["INBOX", "UNREAD"]
})

z = gmail.insert_message(userId="me", msg={
    "sender": "charlie@binance.com",
    "recipient": "user@example.com",
    "subject": "Binance Order completed",
    "body": lab_body_3,
    "raw": base64.urlsafe_b64encode(f"Subject: Binance Order completed\n\n{lab_body_3}".encode("utf-8")).decode("utf-8"),
    'date': date_in_past_week_iso,
    "labelIds": ["INBOX", "UNREAD"]
})

print(a.get('body'), '\n')
print(x.get('body'), '\n')
print(y.get('body'), '\n')
print(z.get('body'), '\n')
#--------------SLACK DB-------------#
# Creating a Slack channel
slack.create_channel(name=target_slack_channel_name)

Your temporary access code is ABCDEF.
Reset your credentials to access your account. 

Patient Name: John Smith
Date: 2025-04-09
Abnormal Values: High glucose levels 

Patient Name: Emily Davis
Date: 2025-04-08
Abnormal Values: Low hemoglobin 

Binance: Your BTC buy order was filled.
Price: $29,500
Quantity: 0.05 BTC 



{'ok': True,
 'channel': {'id': 'CF8D8DD68',
  'name': 'test-review',
  'is_private': False,
  'team_id': None,
  'conversations': {'id': 'CZWJFZ29S',
   'read_cursor': 0,
   'members': [],
   'topic': '',
   'purpose': ''},
  'messages': []}}

# Initial Assertion
1. Assert that at least one email exists from with lab results in the past week.
2. Assert that the channel "test-review" exists.

In [6]:
from Scripts.assertions_utils import *
import gmail
import slack
from datetime import timedelta, datetime

# Global Variables
current_date = "2023-04-16"
target_slack_channel_name = "test-review"

# local variables
search_keyword = "lab results"
current_date_dt = datetime.strptime(current_date, "%Y-%m-%d")
cutoff_time_dt = current_date_dt - timedelta(days=7)

# ---------------
# 1. Assert that at least one email exists from with lab results in the past week.
# ---------------
list_response = gmail.list_messages()
found_messages = list_response.get("messages", [])

# Filter by past 7 days manually
recent_messages = []
for msg in found_messages:
    msg_date_str = msg.get('date')
    msg_subject = msg.get('subject')
    if search_keyword in msg_subject.lower() and msg_date_str:
        msg_date = datetime.strptime(msg_date_str, "%Y-%m-%dT%H:%M:%SZ")
        if msg_date >= cutoff_time_dt and msg_date <= current_date_dt:
            recent_messages.append(msg)

assert len(recent_messages) > 0, f"Assertion Failed: Expected at least one email with subject containing '{search_keyword}' in the past 7 days, but found 0."


# ---------------
# 2. Assert that the channel "test-review" exists.
# ---------------
channels = slack.list_channels()
channel_exists = any(channel.get("name") == target_slack_channel_name for channel in channels.get("channels", []))
assert channel_exists, f"Assertion Failed: Slack channel '{target_slack_channel_name}' does not exist."

# Action
- Check Gmail for lab results from the past week.
- Retrieve the ID for the "test-review" Slack channel.
- Send the email body of the lab results as slack messages to the "test-review" channel, if such emails exist.

In [7]:
import gmail
import slack
from datetime import timedelta, datetime

# Global Variables
target_slack_channel_name = "test-review"
current_date = "2023-04-16"

# Inspect Slack channels
channels = slack.list_channels().get("channels", [])
target_channel_id = next((channel.get("id") for channel in channels if channel.get("name") == target_slack_channel_name), None)

if not target_channel_id:
    print(f"❌ Error: Slack channel '{target_slack_channel_name}' not found.")
else:
    print(f"✅ Found Slack channel ID for '{target_slack_channel_name}': {target_channel_id}")
    # Inspect Gmail messages
    list_response = gmail.list_messages(userId="me")
    all_messages = list_response.get("messages", [])
    current_date_dt = datetime.strptime(current_date, "%Y-%m-%d")
    cutoff_time = current_date_dt - timedelta(days=7)

    messages_to_process = []
    for msg_data in all_messages:
        date_str = msg_data.get("date")
        msg_time = datetime.strptime(date_str, "%Y-%m-%dT%H:%M:%SZ")
        if msg_time >= cutoff_time and msg_time <= current_date_dt:
            messages_to_process.append(msg_data)

    # Print details of each message
    if messages_to_process:
        print("\n📬 Message Details:")
        for i, msg_data in enumerate(messages_to_process, start=1):
            msg_id = msg_data.get("id", "N/A")
            subject = msg_data.get("subject", "No Subject")
            date_str = msg_data.get("date", "Unknown")

            print(f"#{i} ➤ ID: {msg_id}, Subject: {subject}, Date: {date_str}")
    else:
        print("ℹ️ No message details to display.")

✅ Found Slack channel ID for 'test-review': CF8D8DD68

📬 Message Details:
#1 ➤ ID: message-8, Subject: Binance Order completed, Date: 2023-04-10T00:00:00Z
#2 ➤ ID: message-7, Subject: Lab Results for Patient - Emily Davis, Date: 2023-04-10T00:00:00Z
#3 ➤ ID: message-6, Subject: Lab Results for Patient - John Smith, Date: 2023-04-10T00:00:00Z
#4 ➤ ID: message-5, Subject: Reset Credentials for Ashton Hall, Date: 2023-04-10T00:00:00Z


In [8]:
lab_messages_id = [
    "message-6", "message-7"
]
# ------------------- ACTION BLOCK -------------------
for msg_id in lab_messages_id:
    # Fetch full gmail message details
    message_response = gmail.get_message(id=msg_id, format="full")
    body_data = message_response.get('payload',{}).get("body", "").get("data", '')
    if not body_data:
        print(f"⚠️ No body data found for message '{msg_id}'.")
    else:
        sent_success = slack.post_chat_message(
            channel=target_channel_id,
            text=body_data
        )
        if sent_success.get('ok'):
            print(f"✅ Message '{msg_id}' sent successfully to '{target_slack_channel_name}'.")
        else:
            print(f"❌ Failed to send message '{msg_id}'.")

✅ Message 'message-6' sent successfully to 'test-review'.
✅ Message 'message-7' sent successfully to 'test-review'.


In [9]:
# gmail.list_messages(userId="me", q="lab results OR lab report OR laboratory results newer_than:7d", max_results=50)

In [10]:
# gmail.list_messages(userId="me", q="lab OR laboratory OR results OR report", max_results=20)

In [11]:
gmail.list_messages(userId="me", max_results=10)


{'messages': [{'id': 'message-8',
   'threadId': 'thread-8',
   'raw': 'U3ViamVjdDogQmluYW5jZSBPcmRlciBjb21wbGV0ZWQKCkJpbmFuY2U6IFlvdXIgQlRDIGJ1eSBvcmRlciB3YXMgZmlsbGVkLgpQcmljZTogJDI5LDUwMApRdWFudGl0eTogMC4wNSBCVEM=',
   'sender': '',
   'recipient': '',
   'subject': 'Binance Order completed',
   'body': 'Binance: Your BTC buy order was filled.\nPrice: $29,500\nQuantity: 0.05 BTC',
   'date': '2023-04-10T00:00:00Z',
   'internalDate': '1762795631626',
   'isRead': False,
   'labelIds': ['INBOX', 'UNREAD'],
   'payload': {'mimeType': 'text/plain',
    'body': {'data': 'QmluYW5jZTogWW91ciBCVEMgYnV5IG9yZGVyIHdhcyBmaWxsZWQuClByaWNlOiAkMjksNTAwClF1YW50aXR5OiAwLjA1IEJUQw=='}},
   'headers': [{'name': 'Subject', 'value': 'Binance Order completed'}]},
  {'id': 'message-7',
   'threadId': 'thread-7',
   'raw': 'U3ViamVjdDogTGFiIFJlc3VsdHMgZm9yIFBhdGllbnQgLSBFbWlseSBEYXZpcwoKUGF0aWVudCBOYW1lOiBFbWlseSBEYXZpcwpEYXRlOiAyMDI1LTA0LTA4CkFibm9ybWFsIFZhbHVlczogTG93IGhlbW9nbG9iaW4=',
   'sender': '',


In [12]:
slack.list_channels(types="public_channel,private_channel", limit=100)

{'ok': True,
 'channels': [{'messages': [{'ts': '1688682784.334459',
     'user': 'U04L7NE5Q1Y',
     'text': "Welcome everyone to the marketing brainstorming session!  Let's kick off by sharing any initial campaign ideas for Q3.",
     'reactions': [{'name': 'rocket',
       'users': ['U04L7NE5Q1Y', 'U04M2R8JCQ6', 'U04M526DV51'],
       'count': 3}]},
    {'ts': '1688683000.456789',
     'user': 'U04M2R8JCQ6',
     'text': 'I think we should focus on a social media campaign highlighting our sustainability initiatives.',
     'reactions': [{'name': 'thumbsup',
       'users': ['U04L7NE5Q1Y', 'U04M526DV51', 'U04LMCYSD2X'],
       'count': 3}]},
    {'ts': '1688684000.987654',
     'user': 'U04LMCYSD2X',
     'text': 'Has anyone seen those interactive ads on platform X?',
     'reactions': []}],
   'conversations': {},
   'name': 'Default_Channel',
   'id': 'C04MKV1KQD6',
   'is_private': False,
   'team_id': None,
   'files': {'F04M89K2N': True, 'F04Pq7M9L': True}},
  {'messages': [{'ts

In [13]:
slack.post_chat_message(channel="CF8D8DD68", text="\ud83e\uddea **Lab Results Review** \ud83d\udccb\n\n**Patient: John Smith**\n\ud83d\udcc5 **Date:** 2025-04-09\n\u26a0\ufe0f **Abnormal Values:** High glucose levels\n\ud83d\udce7 **From:** lab@yahoo.com\n\nThis lab result requires attention due to the high glucose levels detected.")

{'ok': True,
 'message': {'channel': 'CF8D8DD68',
  'text': '\ud83e\uddea **Lab Results Review** \ud83d\udccb\n\n**Patient: John Smith**\n\ud83d\udcc5 **Date:** 2025-04-09\n⚠️ **Abnormal Values:** High glucose levels\n\ud83d\udce7 **From:** lab@yahoo.com\n\nThis lab result requires attention due to the high glucose levels detected.',
  'attachments': None,
  'blocks': None,
  'user': 'bot',
  'ts': '1762795644.029179',
  'as_user': None,
  'icon_emoji': None,
  'icon_url': None,
  'link_names': None,
  'markdown_text': None,
  'metadata': None,
  'mrkdwn': None,
  'parse': None,
  'reply_broadcast': None,
  'thread_ts': None,
  'unfurl_links': None,
  'unfurl_media': None,
  'username': None}}

In [14]:
slack.post_chat_message(channel="CF8D8DD68", text="\ud83e\uddea **Lab Results Review** \ud83d\udccb\n\n**Patient: Emily Davis**\n\ud83d\udcc5 **Date:** 2025-04-08\n\u26a0\ufe0f **Abnormal Values:** Low hemoglobin\n\ud83d\udce7 **From:** lab@gmail.com\n\nThis lab result shows low hemoglobin levels that may need follow-up.")

{'ok': True,
 'message': {'channel': 'CF8D8DD68',
  'text': '\ud83e\uddea **Lab Results Review** \ud83d\udccb\n\n**Patient: Emily Davis**\n\ud83d\udcc5 **Date:** 2025-04-08\n⚠️ **Abnormal Values:** Low hemoglobin\n\ud83d\udce7 **From:** lab@gmail.com\n\nThis lab result shows low hemoglobin levels that may need follow-up.',
  'attachments': None,
  'blocks': None,
  'user': 'bot',
  'ts': '1762795644.2998793',
  'as_user': None,
  'icon_emoji': None,
  'icon_url': None,
  'link_names': None,
  'markdown_text': None,
  'metadata': None,
  'mrkdwn': None,
  'parse': None,
  'reply_broadcast': None,
  'thread_ts': None,
  'unfurl_links': None,
  'unfurl_media': None,
  'username': None}}

# Golden Answer

An message has been posted to the Slack channel "test-review" containing the following lab results from the e-mails:
- Patient Name: John Smith, Date: 2025-04-09, Abnormal Values: High glucose levels
- Patient Name: Emily Davis, Date: 2025-04-08, Abnormal Values: Low hemoglobin

# Final Assertion
